# Session 5 · Asynchronous · Correlation and linear regression with statsmodels
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/neoma-python-2627/blob/main/notebooks/S05_selfpaced_regression.ipynb)  ·  *Python Programming for Business Analytics · NEOMA 2026/27 · dr. Uroš Godnov*

**Step 6 of the Session 5 mission (about 70 minutes).** No Zoom today: this notebook is your lecturer. Read, run, predict, and write your answers in the 💬 cells. Then continue with the practice notebook and the pay equity memo in Moodle.

**Scenario.** You join the analytics team of a real-estate investment fund. The investment committee wants to know **which house features drive prices** and **how much one extra square metre is worth**. Data: 2,930 house sales in Ames, Iowa (De Cock, 2011), the same data as in the R course (`modeldata::ames`).

| Part | Minutes | Topic |
|---|---|---|
| 1 | 15 | Correlation: strength, uncertainty, traps |
| 2 | 20 | Simple regression: one predictor |
| 3 | 20 | Multiple regression: holding other things constant |
| 4 | 10 | Prediction with an interval, and a paragraph for managers |
| 5 | 5 | R ↔ Python summary |

🆚 Why `statsmodels`? Its **formula interface** is almost identical to R: `lm(price ~ living_m2, data = houses)` becomes `smf.ols("price ~ living_m2", data=houses).fit()`.

In [ ]:
# ⚙️ Setup: run this cell first (works in Google Colab and in Positron)
from pathlib import Path
import numpy as np
import pandas as pd

REPO = "https://raw.githubusercontent.com/urosgodnov/neoma-python-2627/main/"


def data_path(name):
    """Use the local course folder if you have it, otherwise read the file from GitHub."""
    for base in (Path("data"), Path("../data")):
        if (base / name).exists():
            return base / name
    return REPO + "data/" + name


pd.set_option("display.max_columns", 30)
print("Python ready · pandas", pd.__version__)
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sns.set_theme(style="whitegrid", rc={"axes.titleweight": "bold", "axes.titlelocation": "left"})

ames = pd.read_csv(data_path("ames.csv"))
QUALITY_LEVELS = ["Very_Poor", "Poor", "Fair", "Below_Average", "Average",
                  "Above_Average", "Good", "Very_Good", "Excellent"]

houses = pd.DataFrame({
    "price": ames["Sale_Price"],
    "living_m2": ames["Gr_Liv_Area"] * 0.0929,          # square feet -> square metres
    "year_built": ames["Year_Built"],
    "garage_cars": ames["Garage_Cars"],
    "quality": pd.Categorical(ames["Overall_Cond"], categories=QUALITY_LEVELS),
    "neighborhood": ames["Neighborhood"],
    "central_air": ames["Central_Air"],
})
houses.describe().round(1)

🆚 This is exactly the R `transmute()` from the R workbook. The `Categorical` keeps the levels **in quality order**, like an R factor; without it Python would sort them alphabetically.

---
## Part 1 · Correlation

Pearson's *r* measures the strength of a **linear** relationship, from −1 to +1. Spearman's ρ uses ranks: it captures any **monotonic** relationship and resists outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(data=houses, x="living_m2", y="price", alpha=0.25, s=12, ax=axes[0])
axes[0].set(title="Bigger houses sell for more", xlabel="Living area (m²)", ylabel="Price ($)")
sns.scatterplot(data=houses, x="year_built", y="price", alpha=0.25, s=12, ax=axes[1])
axes[1].set(title="Newer houses too, but not in a straight line", xlabel="Year built", ylabel="Price ($)")
plt.show()

In [ ]:
pearson = stats.pearsonr(houses["living_m2"], houses["price"])
ci = pearson.confidence_interval(confidence_level=0.95)
print(f"Pearson r = {pearson.statistic:.3f}, 95% CI [{ci.low:.3f}, {ci.high:.3f}], p = {pearson.pvalue:.1e}")

In [ ]:
for method in ["pearson", "spearman"]:
    print(method, round(houses["year_built"].corr(houses["price"], method=method), 3))

💬 **Your answer:** Spearman is larger than Pearson for `year_built`. What does that tell you about the shape of the relationship?

*…*

### A correlation matrix as a heatmap

In [ ]:
corr = houses[["price", "living_m2", "year_built", "garage_cars"]].corr()
ax = sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
ax.set_title("Living area is the strongest single correlate of price")
plt.show()

### Trap: Simpson's paradox
🔮 **Predict:** is a longer bill associated with a deeper bill in penguins? Positive or negative?

In [ ]:
penguins = pd.read_csv(data_path("penguins.csv")).dropna(subset=["bill_len", "bill_dep"])
print("All penguins:", round(penguins["bill_len"].corr(penguins["bill_dep"]), 2))
print(penguins.groupby("species")[["bill_len", "bill_dep"]].corr().xs("bill_dep", level=1)["bill_len"].round(2))

In [ ]:
g = sns.lmplot(data=penguins, x="bill_len", y="bill_dep", hue="species", height=4, aspect=1.4,
               scatter_kws={"s": 12, "alpha": 0.6})
sns.regplot(data=penguins, x="bill_len", y="bill_dep", scatter=False, color="black",
            line_kws={"ls": "--"}, ax=g.ax)
g.ax.set_title("Overall the slope is negative (dashed); within every species it is positive")
plt.show()

💬 **Your answer:** give one business example where the overall trend could point the wrong way because a group variable is missing.

*…*

---
## Part 2 · Simple regression

A regression line answers *"how much does y change when x increases by one unit?"*:

$$\text{price} = \beta_0 + \beta_1 \cdot \text{living\_m2} + \varepsilon$$

In [ ]:
m1 = smf.ols("price ~ living_m2", data=houses).fit()
print(m1.summary().tables[1])

**Reading the output**
- `coef` of `living_m2`: each extra m² is associated with this many dollars, on average.
- `[0.025  0.975]`: the 95 % confidence interval of the slope.
- `P>|t|`: the p-value (is the slope distinguishable from zero?).

The pieces you need for reports (🆚 `tidy()` and `glance()` from broom):

In [ ]:
def tidy(model, alpha=0.05):
    """A broom-style coefficient table."""
    ci = model.conf_int(alpha=alpha)
    return pd.DataFrame({
        "estimate": model.params, "std_error": model.bse, "p_value": model.pvalues,
        "conf_low": ci[0], "conf_high": ci[1],
    })


def glance(model):
    """A broom-style one-row model summary."""
    return pd.DataFrame({
        "r_squared": [model.rsquared], "adj_r_squared": [model.rsquared_adj],
        "sigma": [np.sqrt(model.scale)], "aic": [model.aic], "n": [int(model.nobs)],
    })


tidy(m1).round(2)

In [ ]:
glance(m1).round(3)

In [ ]:
b = m1.params["living_m2"]
lo, hi = m1.conf_int().loc["living_m2"]
print(
    f"Each additional square metre is associated with a price that is ${b:,.0f} higher on average "
    f"(95% CI ${lo:,.0f} to ${hi:,.0f}). Living area alone explains {m1.rsquared:.0%} of the variation in prices; "
    f"a typical prediction error is about ${np.sqrt(m1.scale):,.0f}."
)

### Outliers with leverage
Three huge houses (> 400 m²) sold for less than $300,000. One point far to the right can tilt the whole line.

In [ ]:
odd = (houses["living_m2"] > 400) & (houses["price"] < 300_000)
m1b = smf.ols("price ~ living_m2", data=houses[~odd]).fit()
print(f"Slope with them: {m1.params['living_m2']:,.0f} $/m² · without them: {m1b.params['living_m2']:,.0f} $/m²")

💬 **Your answer:** should we remove these houses? (Hint: they were partial sales of unfinished houses, all in Edwards, built in 2007–2008.) Argue in two sentences.

*…*

### Diagnostics: is the straight line a fair summary?
🆚 `performance::check_model()`. We check two things: residuals vs fitted values (no pattern, constant spread) and the Q-Q plot (residuals roughly normal).

In [ ]:
def check_model(model, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(model.fittedvalues, model.resid, s=8, alpha=0.3)
    axes[0].axhline(0, color="red", lw=1)
    axes[0].set(title="Residuals vs fitted", xlabel="Fitted values", ylabel="Residuals")
    sm.qqplot(model.resid, line="s", ax=axes[1], markersize=3, alpha=0.4)
    axes[1].set_title("Normal Q-Q")
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    plt.show()


check_model(m1, "price ~ living_m2: the spread grows with price (a funnel)")

The funnel means **heteroscedasticity**: errors are bigger for expensive houses. Prices grow in **percent**, not in dollars, so we model `log(price)`.

In [ ]:
m2 = smf.ols("np.log(price) ~ living_m2", data=houses[~odd]).fit()
check_model(m2, "log(price) ~ living_m2: much more even")
pct = (np.exp(m2.params["living_m2"]) - 1) * 100
print(f"Each extra m² is associated with a {pct:.2f}% higher price; 10 m² ≈ {(np.exp(10 * m2.params['living_m2']) - 1) * 100:.1f}%.")

---
## Part 3 · Multiple regression: holding other things constant

Big houses tend to be newer and have bigger garages. A multiple regression separates the effects: each coefficient is the association **when the other predictors are held constant**.

In [ ]:
m3 = smf.ols(
    "np.log(price) ~ living_m2 + year_built + garage_cars + central_air + quality",
    data=houses[~odd],
).fit()
tidy(m3).round(4)

🔮 **Predict:** what is the reference level of `quality`? It is the **first** category: `Very_Poor`. Every quality coefficient is compared with very poor houses, which are only 7 in the data. Not a useful comparison for a committee.

Change the reference level to `Average` (🆚 `fct_relevel()`):

In [ ]:
m4 = smf.ols(
    "np.log(price) ~ living_m2 + year_built + garage_cars + central_air + C(quality, Treatment('Average'))",
    data=houses[~odd],
).fit()
good = m4.params["C(quality, Treatment('Average'))[T.Good]"]
print(f"Good condition vs Average, same size, age, garage and air: {(np.exp(good) - 1) * 100:+.1f}% price premium")

In [ ]:
comparison = pd.concat(
    {"m2: living_m2": glance(m2), "m4: + age, garage, air, quality": glance(m4)}
).droplevel(1)
comparison.round(3)

Actual vs predicted (🆚 `augment()`): a perfect model would put every point on the dashed line.

In [ ]:
aug = houses[~odd].assign(predicted=np.exp(m4.fittedvalues))
ax = sns.scatterplot(data=aug, x="predicted", y="price", alpha=0.25, s=10)
lims = [0, aug[["price", "predicted"]].max().max()]
ax.plot(lims, lims, ls="--", color="black")
ax.set(title=f"The model explains {m4.rsquared:.0%} of the variation in log prices",
       xlabel="Predicted price ($)", ylabel="Actual price ($)")
plt.show()

⚠️ **Correlation is not causation.** The committee may read *"central air adds 10 %"* as *"install air conditioning and earn 10 %"*. The data cannot show that: houses with central air differ in many unmeasured ways.

---
## Part 4 · Prediction with an interval
A **confidence** interval is about the *average* house with these features. A **prediction** interval is about *one* house: it is much wider. For a buyer, the prediction interval is the honest one.

In [ ]:
new_house = pd.DataFrame({
    "living_m2": [120], "year_built": [1995], "garage_cars": [1],
    "central_air": ["Y"], "quality": pd.Categorical(["Good"], categories=QUALITY_LEVELS),
})
pred = m4.get_prediction(new_house).summary_frame(alpha=0.10)      # 90% intervals
pred_dollars = np.exp(pred[["mean", "mean_ci_lower", "mean_ci_upper", "obs_ci_lower", "obs_ci_upper"]])
pred_dollars.round(-3)

In [ ]:
p = pred_dollars.iloc[0]
print(
    f"For a 120 m² house built in 1995 with one garage space, central air and good condition, the model predicts "
    f"about ${p['mean']:,.0f}. Ninety percent of such houses should sell between ${p['obs_ci_lower']:,.0f} and "
    f"${p['obs_ci_upper']:,.0f}. The model explains {m4.rsquared:.0%} of the variation in log prices, so location "
    f"and finish, which it ignores, still matter. Treat the figure as a starting point for negotiation, not a valuation."
)

💬 **Your answer:** why is the prediction interval so much wider than the confidence interval?

*…*

---
## Part 5 · R ↔ Python summary

| Task | R | Python |
|---|---|---|
| Correlation with CI | `cor.test(x, y) \|> tidy()` | `stats.pearsonr(x, y).confidence_interval()` |
| Spearman | `cor(x, y, method = "spearman")` | `x.corr(y, method="spearman")` |
| Fit | `lm(y ~ x1 + x2, data = df)` | `smf.ols("y ~ x1 + x2", data=df).fit()` |
| Log outcome | `lm(log(y) ~ x)` | `smf.ols("np.log(y) ~ x")` |
| Reference level | `fct_relevel(f, "Average")` | `C(f, Treatment("Average"))` |
| Coefficients + CI | `tidy(m, conf.int = TRUE)` | `m.params`, `m.conf_int()` (or our `tidy()`) |
| R², sigma, AIC | `glance(m)` | `m.rsquared`, `np.sqrt(m.scale)`, `m.aic` |
| Fitted, residuals | `augment(m)` | `m.fittedvalues`, `m.resid` |
| Diagnostics | `performance::check_model(m)` | residual plot + `sm.qqplot()` |
| Prediction interval | `predict(m, new, interval = "prediction", level = 0.9)` | `m.get_prediction(new).summary_frame(alpha=0.1)` |

**Next steps in Moodle:** Step 6b practice notebook → Step 7 pay equity memo → Step 8 chart gallery → Step 9 exit ticket.